# Simplified setting: reduced-state, frozen-readout, gauge-free attention

Reduces the ICL Markov-with-noise task (folders 6/10) to a smaller, more theoretically
tractable version, while preserving the properties that make the FFN's finite-sample
count-denoising story visible. Everything is controlled from the **CONFIG cell** below —
change a setting and re-run top to bottom.

| change | folder 10 | here | why it's safe |
|---|---|---|---|
| vocab | V_sig=10, n_noise=10 | **V_sig=4, n_noise=4** | still 2-successors-per-state (sparse prior); *higher* relative sparsity (2/4 vs 2/10) |
| context / loss window | seq_len=200, loss on last 100 | **seq_len=100, loss on last 50** | per-state sample count at window start ≈12.5α (new) vs ≈10α (old, 10 states/200 ctx) — same finite-sample regime |
| successor split | p1~U(0,1) | **p1~U(0.1,0.9)** | avoids near-deterministic states that need no context inference |
| thresholds | 0.1..0.9 | **[0.4,0.5,0.6,0.7,0.8,0.9,1.0]** | drops the too-sparse-signal low end; α=1.0 is a no-noise edge case (qualitatively different — no denoising sub-task at all, kept as an upper-bound sanity check) |
| architecture | separate Wq,Wk,Wv,Wo | **combined W_qk, W_ov** (folder 7 style) | removes the within-head rotation gauge; exact reparametrization, not an approximation |
| trainable params | everything | **W_qk, W_ov (both layers) + fc_out (+bias) only** | isolates "does GD find the right attention circuit" from "does GD find the right readout" |
| FFN | trained | **frozen at a fitted log(f+δ) count-readout**, δ=1 ⇒ exactly the context-Bayes / Beta(1,1) prior | numerically stable (fit + verified below in-notebook), theoretically principled — not copied stale tuned constants |
| optimizer | Adam only | **Adam or SGD**, one switch in CONFIG | vanilla SGD matches the GD-learns-causal-structure theory (Nichani et al. 2024) |
| data | 20k train | **10k train** | plenty given fewer states / shorter context |

**Frozen-readout design.** The FFN (`ff1`,`ff2`) is overwritten with a *fitted*, block-diagonal
GELU construction implementing, independently per signal state `v`, the residual correction
`f_v ↦ f_v + CORE(f_v) ≈ log(f_v + δ)` at coordinate `in2+v` of the hidden state (`in2` = the
offset of layer-2's attention-output block — the same coordinate convention the project's
hand-built W* construction has always used). All other coordinates (incl. the noise block) are
left untouched; the constant noise floor `(1-α)/n_noise` is left for the *trainable* `fc_out.bias`
to absorb directly, no FFN needed for it. `CORE` is fit **once at notebook load** via a
closed-form spline warm-start + a short Adam refinement (see the "frozen-readout fit" cells) — it
is not a hardcoded constant, so it re-derives itself correctly if you change `V_SIG`, `FREEZE_DELTA`,
or anything else.

This gives GD a *fixed, correct* log-count readout for free; the only question left for gradient
descent is whether it can find the attention circuit that feeds count information into the right
coordinate — a much narrower, more tractable question than the joint problem folders 6–10 study.


In [2]:
# ── imports + shared-box temp-dir routing (this box's system /tmp is unusable) ──
import json, os, time, math
import tempfile

_TMP = os.path.join(os.path.expanduser('~'), '.cache', 'ashu_tmp')
os.makedirs(_TMP, exist_ok=True)
for _v in ('TMPDIR', 'TMP', 'TEMP'):
    os.environ[_v] = _TMP
tempfile.tempdir = _TMP
os.environ.setdefault('MPLCONFIGDIR', _TMP)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt


In [3]:
# ══════════════════════════════ CONFIG ══════════════════════════════════════
# Every setting the experiment depends on lives here. Change anything and re-run
# top to bottom -- nothing downstream is hardcoded against these values.

# ── task size ────────────────────────────────────────────────────────────────
V_SIG   = 4                      # number of "ruled" (signal) states, each with 2 successors
N_NOISE = 4                      # number of noise tokens
V       = V_SIG + N_NOISE        # total vocab

SEQ_LEN     = 100                # total context length per sequence
TOKEN_START = 50                 # loss / eval window: last (SEQ_LEN - TOKEN_START) tokens
TOKEN_END   = SEQ_LEN

# NOTE: literal list from the spec discussion; the phrasing "removing thresholds < 0.5" was
# inconsistent with keeping 0.4 in the list -- this is the literal list, edit freely.
# alpha=1.0 is a no-noise edge case: qualitatively different (no denoising sub-task at all),
# kept here only as an upper-bound sanity check, not as "just another point on the curve".
THRESHOLDS = [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

SUCC_P_LOW, SUCC_P_HIGH = 0.1, 0.9   # per-state successor split p1~U(low,high), p2=1-p1
                                      # (clipped away from 0/1 so every state needs real inference)

# ── data ────────────────────────────────────────────────────────────────────
DATA_DIR  = 'dataset'             # written by data_generation.py -- run that script first
N_TRAIN   = 10_000
N_TEST    = 2_000
# No held-out validation split: trains on the FULL N_TRAIN, checkpoints on best TEST loss
# (the only held-out signal left). Fine for this toy/mech-interp setting (studying
# optimization dynamics, not an unbiased generalization estimate).
DATA_SEED = 42

# ── architecture ─────────────────────────────────────────────────────────────
ATTN_DIM = V + SEQ_LEN           # = initial_dim, same convention as folders 6/8/10's W*
                                  # (keeps the "frequency lives at in2+v" coordinate valid)
BIAS     = True                  # fc_out bias (needed to absorb the noise floor)

# ── frozen log-readout construction ──────────────────────────────────────────
FREEZE_READOUT   = True          # freeze ff1/ff2 at the fitted log(f+delta) count-readout
FREEZE_DELTA     = 1.0           # Laplace / Beta(1,1) smoothing constant
                                  # (=1.0 makes this EXACTLY the context-Bayes prior, see CE plot)
FREEZE_HIDDEN_K  = 12            # GELU hidden units per signal state in the frozen block
FREEZE_FIT_FMAX  = 40.0          # range over which log(f+delta) is fit (max plausible per-state count)
FREEZE_FIT_STEPS = 2000          # Adam refinement steps after the closed-form spline warm-start

# ── which parameters receive gradients ───────────────────────────────────────
# 'attn'   -> layer1.W_qk, layer1.W_ov, layer2.W_qk, layer2.W_ov
# 'ffn'    -> ff1, ff2   (ignored / forced-frozen whenever FREEZE_READOUT=True)
# 'fc_out' -> fc_out.weight, fc_out.bias
TRAIN_PARAMS = ['attn', 'fc_out']

# ── optimization ──────────────────────────────────────────────────────────────
OPTIMIZER = 'adam'               # 'adam' or 'sgd' -- flip this one switch
LR = {'adam': 1e-3, 'sgd': 0.1}[OPTIMIZER]   # sane per-optimizer default; override directly if needed
MOMENTUM     = 0.9               # only used when OPTIMIZER='sgd'
WEIGHT_DECAY = 0.0
GRAD_CLIP    = 0.5
EPOCHS       = 200
BATCH_SIZE   = 256
PATIENCE     = 60                # early-stop patience (epochs with no test-loss improvement)
EVAL_BATCH   = 256               # mini-batch size for eval forward passes (bounds peak GPU memory
                                  # on a contested shared box -- see CLAUDE.md §6 gotchas)
SEED = 7

# ── misc ──────────────────────────────────────────────────────────────────────
DEVICE     = 'cuda:0'
MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

device = torch.device(DEVICE if torch.cuda.is_available() else 'cpu')
print(f'device={device}   V_sig={V_SIG} n_noise={N_NOISE} V={V}   seq_len={SEQ_LEN}  '
      f'window=[{TOKEN_START}:{TOKEN_END}]')
print(f'thresholds={THRESHOLDS}')
print(f'FREEZE_READOUT={FREEZE_READOUT} (delta={FREEZE_DELTA})   TRAIN_PARAMS={TRAIN_PARAMS}   '
      f'optimizer={OPTIMIZER} lr={LR}')

_CFG_KEYS = ['V_SIG', 'N_NOISE', 'V', 'SEQ_LEN', 'TOKEN_START', 'TOKEN_END', 'THRESHOLDS',
             'SUCC_P_LOW', 'SUCC_P_HIGH', 'N_TRAIN', 'N_TEST', 'DATA_SEED',
             'ATTN_DIM', 'BIAS', 'FREEZE_READOUT', 'FREEZE_DELTA', 'FREEZE_HIDDEN_K',
             'FREEZE_FIT_FMAX', 'FREEZE_FIT_STEPS', 'TRAIN_PARAMS', 'OPTIMIZER', 'LR',
             'MOMENTUM', 'WEIGHT_DECAY', 'GRAD_CLIP', 'EPOCHS', 'BATCH_SIZE', 'PATIENCE', 'SEED']


device=cuda:0   V_sig=4 n_noise=4 V=8   seq_len=100  window=[50:100]
thresholds=[0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
FREEZE_READOUT=True (delta=1.0)   TRAIN_PARAMS=['attn', 'fc_out']   optimizer=adam lr=0.001


## Data

Data is **pre-generated on disk** by `data_generation.py` (run it once before this notebook --
same convention as folders 6/10) into `dataset/{train,test}/data_x_{alpha}.pt`. No separate
validation file: validation is carved out of the train blob at *training* time via a
fixed-seed permutation (`split_train_val`, below) -- the same approach `train_identical_arch.py`
(folder 10) uses, so `models/metrics_*.json` always records the exact train/val split used.


In [4]:
def load_data(split, thr, n_cap=None):
    """Load a threshold's pre-generated blob written by data_generation.py. No on-the-fly
    generation -- this notebook trains/evaluates on the EXACT sequences on disk, so results are
    reproducible across runs/kernels without needing to reseed a generator in-notebook."""
    path = os.path.join(DATA_DIR, split, f'data_x_{thr}.pt')
    blob = torch.load(path, weights_only=False)
    if n_cap is not None:
        n = min(n_cap, blob['x'].shape[0])
        blob = dict(blob)
        blob['x'] = blob['x'][:n]; blob['y'] = blob['y'][:n]
        for k in ('transitions', 'trans_probs'):
            if k in blob: blob[k] = blob[k][:n]
    return blob


## Model: combined-QK/OV attention + (frozen) log-readout FFN

In [5]:
class CombinedAttentionBlock(nn.Module):
    """QK/OV combined parametrization (folder 7 style, project CLAUDE.md §3):
    scores = x @ W_qk @ x^T ;  out = softmax(scores) @ (x @ W_ov).
    W_qk, W_ov ARE the only free parameters -- this removes the within-head Q/K and V/O
    rotation gauge (exact, not an approximation; folder 7 verified collapsing a hand-built
    W* into this form gives 0.0 logit diff vs the separate-matrix parametrization)."""
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.output_dim = output_dim
        self.W_qk = nn.Parameter(torch.empty(input_dim, input_dim))
        self.W_ov = nn.Parameter(torch.empty(input_dim, output_dim))
        nn.init.xavier_uniform_(self.W_qk)
        nn.init.xavier_uniform_(self.W_ov)

    def forward(self, x, mask=None):
        scores = torch.bmm(x @ self.W_qk, x.transpose(1, 2)) / (self.output_dim ** 0.5)
        if mask is not None:
            scores = scores + mask
        attn_out = torch.bmm(F.softmax(scores, dim=-1), x @ self.W_ov)
        return torch.cat([x, attn_out], dim=-1), scores


class SimplifiedTransformerMLP(nn.Module):
    """Two CombinedAttentionBlock layers + a position-wise GELU FFN head + linear unembedding.
    Same skeleton as folders 6/8/10's DisentangledTransformerMLP, but attention is the combined
    (W_qk, W_ov) parametrization and the FFN is (optionally) frozen -- see build_frozen_ffn().
    `in2` marks where layer-2's attention-output block begins in the concatenated hidden state --
    the same coordinate convention the project's hand-built W* construction has always used for
    "where the frequency counts live"."""
    def __init__(self, total_vocab, seq_len, attn_dim, v_sig, ffn_hidden_per_state, bias=True):
        super().__init__()
        self.total_vocab = total_vocab
        self.seq_len      = seq_len
        self.v_sig        = v_sig
        self.initial_dim  = total_vocab + seq_len
        self.layer1 = CombinedAttentionBlock(self.initial_dim, attn_dim)
        self.layer2 = CombinedAttentionBlock(self.initial_dim + attn_dim, attn_dim)
        self.D   = self.initial_dim + 2 * attn_dim
        self.in2 = self.initial_dim + attn_dim
        self.ff1 = nn.Linear(self.D, v_sig * ffn_hidden_per_state)
        self.ff2 = nn.Linear(v_sig * ffn_hidden_per_state, self.D)
        self.act = nn.GELU()
        self.fc_out = nn.Linear(self.D, total_vocab, bias=bias)

    def forward(self, x):
        B, T = x.shape
        tok = F.one_hot(x, num_classes=self.total_vocab).float()
        pos = F.one_hot(torch.arange(T, device=x.device),
                        num_classes=self.seq_len).float().unsqueeze(0).expand(B, T, -1)
        s = torch.cat([tok, pos], dim=-1)
        mask = torch.triu(torch.ones(T, T, device=x.device) * float('-inf'), diagonal=1)
        s, _ = self.layer1(s, mask)
        s, _ = self.layer2(s, mask)
        s = s + self.ff2(self.act(self.ff1(s)))
        return self.fc_out(s)


## Frozen log-readout: fit + install

`fit_log_core` fits `CORE(f) = GELU(f·a+b) @ w + c` to the residual target `log(f+δ) − f`, so that
after the FFN's residual add, coordinate `in2+v` holds `f_v + CORE(f_v) ≈ log(f_v+δ)`. Uses a
closed-form spline warm-start (GELU breakpoints spread evenly over `[0, FREEZE_FIT_FMAX]`, then a
linear least-squares solve for `w,c` with `a,b` held fixed) followed by a short joint Adam
refinement — this converges to a much tighter fit than random init + pure gradient descent.

`build_frozen_ffn` then installs this fit **block-diagonally**: state `v`'s `FREEZE_HIDDEN_K`
hidden units read *only* coordinate `in2+v` and write back *only* to `in2+v`. Every other
coordinate (incl. the whole noise block) is left untouched — the noise floor `(1-α)/n_noise` is a
context-independent constant, so it's left for the trainable `fc_out.bias` to learn directly;
routing it through the FFN would be redundant.

In [6]:
def fit_log_core(delta=FREEZE_DELTA, k=FREEZE_HIDDEN_K, fmax=FREEZE_FIT_FMAX,
                  steps=FREEZE_FIT_STEPS, device='cpu', seed=0):
    torch.manual_seed(seed)
    f = torch.linspace(0, fmax, 400, device=device).unsqueeze(1)          # [N,1]
    target = torch.log(f + delta) - f                                    # [N,1]

    # closed-form spline warm-start: spread GELU breakpoints evenly over [0,fmax],
    # then solve for (w,c) via linear least squares with (a,b) held fixed.
    breakpoints = torch.linspace(0, fmax, k, device=device)
    a0 = torch.full((1, k), 6.0 / max(fmax, 1e-6), device=device)
    b0 = (-a0.squeeze(0) * breakpoints).unsqueeze(0)
    a = nn.Parameter(a0.clone()); b = nn.Parameter(b0.clone())
    with torch.no_grad():
        basis  = F.gelu(f * a + b)
        basis1 = torch.cat([basis, torch.ones(basis.shape[0], 1, device=device)], dim=1)
        sol    = torch.linalg.lstsq(basis1, target).solution
        w0, c0 = sol[:k], sol[k:k + 1].squeeze(0)
    w = nn.Parameter(w0.clone()); c = nn.Parameter(c0.clone())

    opt = torch.optim.Adam([a, b, w, c], lr=1e-2)
    for _ in range(steps):
        opt.zero_grad()
        pred = F.gelu(f * a + b) @ w + c
        loss = F.mse_loss(pred, target)
        loss.backward(); opt.step()
    return a.detach(), b.detach(), w.detach(), c.detach(), float(loss.item())


def build_frozen_ffn(model, delta=FREEZE_DELTA, k=FREEZE_HIDDEN_K, fmax=FREEZE_FIT_FMAX,
                      steps=FREEZE_FIT_STEPS):
    dev = next(model.parameters()).device
    a, b, w, c, mse = fit_log_core(delta, k, fmax, steps, device=dev)
    Vs, D, in2 = model.v_sig, model.D, model.in2
    W1 = torch.zeros(Vs * k, D, device=dev); B1 = torch.zeros(Vs * k, device=dev)
    W2 = torch.zeros(D, Vs * k, device=dev); B2 = torch.zeros(D, device=dev)
    for v in range(Vs):
        rows = slice(v * k, (v + 1) * k)
        W1[rows, in2 + v] = a.squeeze(0)
        B1[rows]          = b.squeeze(0)
        W2[in2 + v, rows] = w.squeeze(1)
        B2[in2 + v]       = c
    with torch.no_grad():
        model.ff1.weight.copy_(W1); model.ff1.bias.copy_(B1)
        model.ff2.weight.copy_(W2); model.ff2.bias.copy_(B2)
    for p in (model.ff1.weight, model.ff1.bias, model.ff2.weight, model.ff2.bias):
        p.requires_grad_(False)
    print(f'[frozen readout] fit MSE={mse:.2e}  over f in [0,{fmax}]  (delta={delta}, '
          f'K={k}/state, {Vs} states)')
    return mse


In [7]:
# ── verify the fit quality before trusting it ────────────────────────────────
if FREEZE_READOUT:
    _a, _b, _w, _c, _mse = fit_log_core()
    _f = torch.linspace(0, FREEZE_FIT_FMAX, 400).unsqueeze(1)
    _target = torch.log(_f + FREEZE_DELTA) - _f
    _pred = F.gelu(_f * _a + _b) @ _w + _c
    _recon = (_pred + _f).squeeze(-1); _true = torch.log(_f + FREEZE_DELTA).squeeze(-1)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(_f.numpy(), _target.numpy(), lw=2, label='target: log(f+δ) − f')
    axes[0].plot(_f.numpy(), _pred.detach().numpy(), '--', lw=2, label='GELU-core fit')
    axes[0].set_xlabel('raw count f'); axes[0].set_title('residual correction fit')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(_f.numpy(), _true.numpy(), lw=2, label='true log(f+δ)')
    axes[1].plot(_f.numpy(), _recon.detach().numpy(), '--', lw=2, label='f + fitted correction')
    axes[1].set_xlabel('raw count f'); axes[1].set_title('reconstructed log-readout')
    axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    _err = (_recon - _true).abs()
    print(f'fit MSE={_mse:.2e}   max |error|={_err.max().item():.4f}   mean |error|={_err.mean().item():.4f}')
    print('If this is too loose, raise FREEZE_HIDDEN_K and/or FREEZE_FIT_STEPS in the CONFIG cell.')


KeyboardInterrupt: 

## Baseline predictors (no training)

`ideal_alpha` knows the true table + true probabilities (the CE floor). `context_bayes` is
Laplace/Beta(1,1)-smoothed empirical counts — the theoretical global minimum for *any*
context-conditioned predictor; with `eps=FREEZE_DELTA=1.0` this is **exactly** what the frozen
FFN targets, so it's the natural thing to compare the trained model against. `ctx_count` is the
same estimator with a smaller smoothing epsilon (closer to raw empirical frequency).

In [ ]:
def ideal_alpha_dist(blob, alpha):
    x_np = blob['x'].numpy(); trans = blob['transitions']; probs = blob['trans_probs']
    V_ = blob['total_vocab']; Vs = blob['vocab_set1']; n_noise = V_ - Vs
    B = x_np.shape[0]; W = TOKEN_END - TOKEN_START
    trans_arr = np.array([[trans[b][v] for v in range(Vs)] for b in range(B)], dtype=np.int64)
    prob_arr  = np.array([[probs[b][v] for v in range(Vs)] for b in range(B)], dtype=np.float64)
    p_win = np.zeros((B, W, V_), dtype=np.float64)
    last_sig = np.zeros(B, dtype=np.int64)
    for t in range(TOKEN_END):
        is_sig = x_np[:, t] < Vs
        last_sig = np.where(is_sig, x_np[:, t], last_sig)
        if t < TOKEN_START: continue
        s0 = trans_arr[np.arange(B), last_sig, 0]; s1 = trans_arr[np.arange(B), last_sig, 1]
        q0 = prob_arr[np.arange(B), last_sig, 0];  q1 = prob_arr[np.arange(B), last_sig, 1]
        p = np.zeros((B, V_), dtype=np.float64)
        p[:, Vs:] = (1.0 - alpha) / n_noise
        p[np.arange(B), s0] += alpha * q0
        p[np.arange(B), s1] += alpha * q1
        p_win[:, t - TOKEN_START] = p
    return p_win

def context_bayes_dist(blob, alpha, eps=FREEZE_DELTA):
    """Laplace/Beta(1,1)-smoothed counts. eps=FREEZE_DELTA by default so this exactly matches
    what the frozen FFN targets (delta=1 <=> Beta(1,1) prior) -- the natural comparison point
    for the trained model."""
    x_np = blob['x'].numpy()
    V_ = blob['total_vocab']; Vs = blob['vocab_set1']; n_noise = V_ - Vs
    B = x_np.shape[0]; W = TOKEN_END - TOKEN_START
    trans_counts = np.zeros((B, V_, Vs), dtype=np.float64)
    pending      = np.zeros((B, V_), dtype=np.float64)
    last_ruled   = np.full(B, -1, dtype=np.int64)
    def _step(col):
        tok = x_np[:, col]; is_sig = tok < Vs
        ri = np.where(is_sig)[0]
        if len(ri):
            rt = tok[ri]
            oh = (np.arange(Vs) == rt[:, None]).astype(np.float64)
            np.add.at(trans_counts, ri, pending[ri, :, None] * oh[:, None, :])
            pending[ri, :] = 0.0; pending[ri, rt] = 1.0; last_ruled[ri] = rt
        ni = np.where(~is_sig)[0]
        if len(ni): pending[ni, tok[ni]] += 1.0
    for i in range(TOKEN_START): _step(i)
    p_win = np.zeros((B, W, V_), dtype=np.float64)
    for t in range(TOKEN_START, TOKEN_END):
        v = x_np[:, t]; is_sig_q = v < Vs
        q   = np.where(is_sig_q, v, np.maximum(last_ruled, 0))
        cnt = trans_counts[np.arange(B), q] + eps
        tot = cnt.sum(axis=1, keepdims=True)
        p = np.zeros((B, V_), dtype=np.float64)
        p[:, :Vs] = cnt / tot * alpha
        p[:, Vs:] = (1.0 - alpha) / n_noise
        p_win[:, t - TOKEN_START] = p
        _step(t)
    return p_win

def ctx_count_dist(blob, alpha, eps=0.01):
    return context_bayes_dist(blob, alpha, eps=eps)   # identical estimator, smaller smoothing eps

def _ce_from_win(p_win, y_np):
    yw = y_np[:, TOKEN_START:TOKEN_END]; B, W = yw.shape
    bb = np.arange(B)[:, None]; ww = np.arange(W)[None, :]
    return float((-np.log(np.maximum(p_win[bb, ww, yw], 1e-12))).mean())


## Analysis-time tooling

Training now happens in **`train.py`** (run standalone, in tmux -- see the next section), not in
this notebook, and trains on the FULL train set (no held-out validation split -- see CONFIG).
`eval_ce` is kept here anyway (cheap, high future value): it lets you re-evaluate a loaded model
on any split later without needing to touch `train.py` again.


In [ ]:
@torch.no_grad()
def eval_ce(model, x, y, batch=EVAL_BATCH):
    model.eval()
    total_nll, total_n = 0.0, 0
    for i in range(0, x.shape[0], batch):
        xb = x[i:i + batch].to(device)
        yb = y[i:i + batch, TOKEN_START:TOKEN_END].to(device)
        logits = model(xb)[:, TOKEN_START:TOKEN_END]
        loss = F.cross_entropy(logits.reshape(-1, V), yb.reshape(-1), reduction='sum')
        total_nll += loss.item(); total_n += yb.numel()
        del xb, yb, logits, loss
        if device.type == 'cuda': torch.cuda.empty_cache()
    model.train()
    return total_nll / total_n


## Train (standalone script) + load results

Training runs in **`train.py`**, not in this notebook -- so a dropped SSH connection doesn't kill
a multi-hour run. In a tmux session, from this folder:

```
tmux new -s simplified_train
python -u train.py
```

`train.py`'s `CONFIG` dict must stay in sync with this notebook's CONFIG cell (same convention as
`data_generation.py`) -- same `V_SIG/N_NOISE/SEQ_LEN/FREEZE_*` etc., since the model's shape
depends on them. It saves, per threshold, into `MODELS_DIR`:

```
model_simplified_{OPTIMIZER}_x_{alpha}.pth      <- checkpoint (best TRAIN loss; no val split)
metrics_simplified_{OPTIMIZER}_x_{alpha}.json   <- full history + config used
```

Ctrl+C during a threshold's epoch loop saves that threshold's best-so-far and moves on to the
next one (same convention as folder 10's `train_identical_arch.py`).

The cell below just **loads** whatever checkpoints exist on disk -- re-run it any time after
`train.py` finishes (or partially finishes) more thresholds, no need to touch anything else.


In [ ]:
def load_trained_model(thr, tag=None):
    """Build a fresh SimplifiedTransformerMLP and load the checkpoint train.py wrote for this
    threshold. Returns (model, metrics_dict) or (None, None) if train.py hasn't reached this
    threshold yet. Kept as a standalone helper (not just inlined in the loop below) since it's
    the entry point for any future analysis cell that needs a live model, not just best_train."""
    tag = tag or f'simplified_{OPTIMIZER}'
    mp = os.path.join(MODELS_DIR, f'model_{tag}_x_{thr}.pth')
    cp = os.path.join(MODELS_DIR, f'metrics_{tag}_x_{thr}.json')
    if not os.path.exists(mp):
        return None, None
    model = SimplifiedTransformerMLP(V, SEQ_LEN, ATTN_DIM, V_SIG, FREEZE_HIDDEN_K, bias=BIAS).to(device)
    model.load_state_dict(torch.load(mp, map_location=device, weights_only=True))
    model.eval()
    metrics = json.load(open(cp)) if os.path.exists(cp) else None
    return model, metrics


results = {}
for thr in THRESHOLDS:
    model, metrics = load_trained_model(thr)
    if model is None:
        print(f'  [skip] thr={thr}: no checkpoint yet (train.py has not reached it) -- '
              f'run/continue train.py in tmux')
        continue
    results[thr] = dict(model=model, history=metrics['history'],
                         best_train=metrics['best_train'], test_at_best=metrics['test_at_best'])
    print(f'  [loaded] thr={thr}  best_train={metrics["best_train"]:.4f}  '
          f'test_at_best={metrics["test_at_best"]:.4f}  '
          f'epochs_run={metrics.get("epochs_run", len(metrics["history"]["test_loss"]))}')

print(f'\n{len(results)}/{len(THRESHOLDS)} thresholds loaded.')
if len(results) < len(THRESHOLDS):
    print('Missing thresholds are silently skipped by the plot cells below -- '
          're-run this cell after train.py trains more of them.')


## Plots: CE vs α, and training curves

In [ ]:
plt.figure(figsize=(8, 5))
ideal_ce, bayes_ce, ctx_ce, gd_ce = [], [], [], []
for thr in THRESHOLDS:
    blob = load_data('test', thr)
    y_np = blob['y'].numpy()
    ideal_ce.append(_ce_from_win(ideal_alpha_dist(blob, thr), y_np))
    bayes_ce.append(_ce_from_win(context_bayes_dist(blob, thr, eps=FREEZE_DELTA), y_np))
    ctx_ce.append(_ce_from_win(ctx_count_dist(blob, thr), y_np))
    gd_ce.append(results[thr]['test_at_best'])

plt.plot(THRESHOLDS, ideal_ce, 'o-', color='tab:green', label='ideal-alpha (true table+probs)')
plt.plot(THRESHOLDS, bayes_ce, 's-', color='tab:blue',  label=f'context-Bayes (eps={FREEZE_DELTA})')
plt.plot(THRESHOLDS, ctx_ce,   '^-', color='tab:cyan',  label='ctx-count (eps=0.01)')
plt.plot(THRESHOLDS, gd_ce,    'P-', color='tab:purple',
         label=f'GD trained ({OPTIMIZER}, frozen_readout={FREEZE_READOUT}, selected by train loss)')
plt.xlabel(r'$\alpha$ (signal probability)'); plt.ylabel('cross-entropy (nats)')
plt.title('Simplified setting: CE vs $\\alpha$')
plt.grid(alpha=0.3); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'ce_vs_alpha_simplified.png'), dpi=130)
plt.show()

print(f'{"alpha":>6}  {"ideal":>8}  {"Bayes":>8}  {"ctx":>8}  {"GD":>8}   GD-Bayes gap')
for i, thr in enumerate(THRESHOLDS):
    print(f'{thr:6.2f}  {ideal_ce[i]:8.4f}  {bayes_ce[i]:8.4f}  {ctx_ce[i]:8.4f}  '
          f'{gd_ce[i]:8.4f}   {gd_ce[i]-bayes_ce[i]:+.4f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for thr in THRESHOLDS:
    h = results[thr]['history']
    axes[0].plot(h['test_loss'], label=f'a={thr}')
    axes[1].plot(h['train_loss'], label=f'a={thr}')
axes[0].set_title('Test CE vs epoch'); axes[0].set_xlabel('epoch')
axes[1].set_title('Train CE vs epoch'); axes[1].set_xlabel('epoch')
for ax in axes:
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'training_curves_simplified.png'), dpi=150)
plt.show()


## Notes for extending this

- **Compare SGD vs Adam**: set `OPTIMIZER='sgd'` in CONFIG and re-run from the training-infra cell
  down — checkpoints are tagged `simplified_{OPTIMIZER}` so both runs coexist on disk.
- **Compare frozen vs trainable FFN**: set `FREEZE_READOUT=False` and add `'ffn'` to `TRAIN_PARAMS`
  to get the fully-joint baseline (closer to folders 6/10's setting, just at the reduced size).
- **If the trained curve doesn't approach context-Bayes**: check the frozen-readout fit-quality
  plot first (mean/max abs error) — a loose fit caps how close GD can get even with perfect
  attention. Raise `FREEZE_HIDDEN_K` / `FREEZE_FIT_STEPS` if needed.
- **Everything is reproducible from CONFIG alone** — no cell here depends on a previous run's
  in-memory state beyond `results` (built fresh each full run) and the frozen-fit cache (re-fit
  each time `build_frozen_ffn` is called, deterministic given `FREEZE_DELTA/K/FMAX/STEPS`).
